# Multi-period optimization

A single-period optimizer answers *"what should I hold?"* from a snapshot, knowing nothing about what you hold today — so costs can only be subtracted afterwards.

`multi_period_mean_variance` answers **"what should I do today?"**: it solves for the whole weight path at once, with trading costs priced inside the objective.

$$\min_{W} \sum_{t=1}^{T} \Big[ -w_t^\top \mu + \tfrac{\gamma}{2} w_t^\top \Sigma w_t + c_{\text{lin}} |w_t - w_{t-1}| + c_{\text{quad}} (w_t - w_{t-1})^2 \Big], \qquad w_0 = w_{\text{prev}}$$

In [1]:
import numpy as np
import pandas as pd

import jaxfolio as jf

returns = jf.generate_returns(n_assets=8, n_days=500, seed=42)
assets = list(returns.columns)
returns.tail(3)

,ASSET_00,ASSET_01,ASSET_02,ASSET_03,ASSET_04,ASSET_05,ASSET_06,ASSET_07
2022-11-29,-0.019948,0.017614,-0.002813,0.011771,0.003112,0.054379,0.016312,0.011695
2022-11-30,0.037018,0.022356,0.001690,0.020355,0.000707,0.014708,-0.026779,0.000998
2022-12-01,-0.021113,-0.019602,-0.019085,-0.002069,-0.002215,0.006448,-0.022377,-0.020116


## The setup

Start with everything in the one name the myopic optimizer wants least, so there is a large position to unwind.

In [2]:
GAMMA = 3.0

myopic = jf.mean_variance(returns, risk_aversion=GAMMA).weights
worst = int(np.argmin(myopic))

holdings = np.zeros(len(assets))
holdings[worst] = 1.0

print(f"holding 100% {assets[worst]}; myopic target wants {myopic[worst]:.1%}")
print(f"myopic turnover to get there: {np.abs(myopic - holdings).sum():.2f}")

holding 100% ASSET_00; myopic target wants 0.0%
myopic turnover to get there: 2.00


## Solve for the path

`weights` is the **first** row — what to trade into today. `trajectory` is the full plan.

In [3]:
res = jf.multi_period_mean_variance(
    returns,
    horizon=10,
    w_prev=holdings,
    risk_aversion=GAMMA,
    costs=jf.TradingCosts(spread_bps=10.0, impact_bps=100.0),
)

path = pd.DataFrame(res.trajectory, columns=assets, index=range(1, 11))
path.index.name = "period"
path.round(3)

,ASSET_00,ASSET_01,ASSET_02,ASSET_03,ASSET_04,ASSET_05,ASSET_06,ASSET_07
period,,,,,,,,
1,0.730,0.041,0.0,0.0,0.104,0.125,0.0,0.0
2,0.527,0.072,0.0,0.0,0.192,0.209,0.0,0.0
3,0.377,0.096,0.0,0.0,0.263,0.263,0.0,0.0
4,0.268,0.113,0.0,0.0,0.320,0.299,0.0,0.0
5,0.193,0.123,0.0,0.0,0.363,0.320,0.0,0.0
6,0.147,0.128,0.0,0.0,0.394,0.331,0.0,0.0
7,0.125,0.128,0.0,0.0,0.413,0.334,0.0,0.0
8,0.123,0.128,0.0,0.0,0.415,0.334,0.0,0.0
9,0.123,0.128,0.0,0.0,0.415,0.334,0.0,0.0


In [ ]:
from jaxfolio import viz

# `myopic` is the frictionless target — a genuine external reference for what a
# *single* rebalance would cost. The plan's own terminal weights are not: for a
# monotone path, total turnover equals |w_T - w_prev| identically.
viz.plot_turnover_schedule(res, reference_weights=myopic);

Trades decay geometrically — a *glide path*. Cumulative turnover lands below what a single rebalance to the myopic target would have cost.

Now the same plan as weights:

In [ ]:
from jaxfolio import viz

viz.plot_weight_path(res);

## Why impact is the term that matters

This is the one non-obvious point. A **linear** cost creates a *no-trade region* — it makes you trade less — but gives no reason to trade *later*, because moving from $a$ to $b$ in one step costs the same as in five. Only **quadratic impact** rewards splitting, since two half-trades cost half of one double-trade.

Both behaviors below are the exact optimum (verified against a CVXPY QP).

In [ ]:
regimes = {
    "frictionless": jf.TradingCosts(),
    "50bps spread only": jf.TradingCosts(spread_bps=50.0),
    "10bps + 100bps impact": jf.TradingCosts(spread_bps=10.0, impact_bps=100.0),
    "10bps + 500bps impact": jf.TradingCosts(spread_bps=10.0, impact_bps=500.0),
}

comparison = {
    name: jf.multi_period_mean_variance(
        returns, horizon=10, w_prev=holdings, risk_aversion=GAMMA, costs=c
    )
    for name, c in regimes.items()
}

pd.DataFrame(
    {n: r.metadata["turnover_path"] for n, r in comparison.items()}, index=path.index
).T.round(3)

In [ ]:
viz.plot_cost_comparison(comparison);

The frictionless and spread-only lines spike at period 1 and then flatline — all the trading happens immediately. The two impact lines decay across the horizon.

So: **if you want execution scheduling, set `impact_bps`.** Setting `spread_bps` alone asks for a turnover-penalized single-period portfolio, and that is correctly what you get.

## End to end

The backtester hands current holdings to any optimizer declaring `w_prev`, and executes a returned `trajectory` step by step. Both are automatic; use `functools.partial` (not a `lambda`, which hides the parameter).

In [8]:
from functools import partial

from jaxfolio.backtest import compare, metrics_table

results = compare(
    returns,
    {
        "myopic": partial(jf.mean_variance, risk_aversion=GAMMA),
        "multi-period": partial(
            jf.multi_period_mean_variance,
            horizon=10,
            risk_aversion=GAMMA,
            costs=jf.TradingCosts(spread_bps=50.0, impact_bps=500.0),
        ),
    },
    lookback=252,
    rebalance_every=21,
    transaction_cost=0.005,
)

metrics_table(results)[["annual_return", "sharpe", "avg_turnover", "total_cost"]].round(4)

,annual_return,sharpe,avg_turnover,total_cost
myopic,0.0324,0.2528,0.5760,0.0346
multi-period,0.0788,0.5437,0.0281,0.0168


In [ ]:
viz.plot_equity_curves(results);

## Reading the diagnostics

The $L_1$ cost is non-differentiable exactly where the optimum sits (at zero trade). It is Huber-smoothed, and because accuracy is *not* monotone in the smoothing width, the solver walks a ladder of widths and keeps whichever iterate is best under the **exact** objective. These keys let you audit that.

In [10]:
for key in ("smoothing_bias", "selected_stage", "residual", "converged", "iterations"):
    print(f"{key:16s} {res.metadata[key]}")

smoothing_bias   3.7439167499542236e-07
selected_stage   6
residual         8.223656550399028e-06
converged        False
iterations       7813


- **`smoothing_bias`** — how much the smoothing understated the true cost. Non-negative, and tiny here.
- **`selected_stage`** — which rung won; `-1` means "never beat holding still", which is *correct* under high costs.
- **`converged`** — `False` is not necessarily a failure: at high cost the residual plateaus while the answer is exactly right (the correct action is "don't trade", and the residual carries the un-actioned cost gradient). Judge on `smoothing_bias` and the objective.

## Two caveats

1. **Turnover caps are soft.** Costs are priced, not capped — you cannot *guarantee* a turnover budget. A hard cap couples the periods and breaks the structure the solver relies on.
2. **Planned ≠ realized turnover.** `turnover_path` assumes no drift; the backtester drifts weights with returns and then trades from the drifted book.

See the [multi-period guide](../../docs/guide/multiperiod.md) for the full cost model and accuracy measurements.

## One-page report

`multiperiod_dashboard` composes the whole story: KPI strip, the weight path, the execution schedule, convergence, and the cost-regime comparison.

In [ ]:
viz.multiperiod_dashboard(res, comparison=comparison, reference_weights=myopic);